# Sequencing Dataset Builder

This notebook builds `data/sequencing_dataset.csv` by starting from `knn_dataset.csv`, reusing the existing geometry-aware sequence features, and adding extra last-15-minute event summaries from the pressure, run, and shot tables.


In [1]:
from importlib import util
from pathlib import Path
import sys

import pandas as pd


LOOKBACK_MINUTES = 15
RUN_STAGE_VALUES = ["bottom", "middle", "top"]
SHOT_BODY_PART_VALUES = ["left_foot", "right_foot", "head", "other"]
SHOT_TECHNIQUE_VALUES = ["normal", "volley", "other", "overhead_kick", "lob"]
SHOT_PLAY_PATTERN_VALUES = [
    "regular_play",
    "corner_kick",
    "counter_attack",
    "indirect_free_kick",
    "direct_free_kick",
    "throw_in",
    "penalty",
]
PERIOD_OFFSETS = {
    "half_1": 0,
    "half_2": 45,
    "extra_time_1": 90,
    "extra_time_2": 105,
}
CHECKPOINT_TO_CONT_MIN = {
    "H1_15": 15,
    "H1_30": 30,
    "H1_45": 45,
    "H2_15": 60,
    "H2_30": 75,
    "H2_45": 90,
    "ET1_15": 105,
    "ET2_15": 120,
}


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root.")


def load_normalized_table(filename: str) -> pd.DataFrame:
    df = pd.read_csv(DATA_DIR / filename, na_values=NA_TOKENS)
    return normalize_dataframe(df)


def add_event_time(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["event_cont_min"] = out["minute"] + out["period"].map(PERIOD_OFFSETS).astype("int64")
    return out


def last_non_null(series: pd.Series, default_value: str = "none") -> str:
    non_null = series.dropna()
    if non_null.empty:
        return default_value
    return str(non_null.iloc[-1])


def safe_mean(series: pd.Series) -> float:
    if series.empty:
        return 0.0
    return float(series.mean())


def safe_std(series: pd.Series) -> float:
    if series.empty:
        return 0.0
    return float(series.std(ddof=0))


def safe_median(series: pd.Series) -> float:
    if series.empty:
        return 0.0
    return float(series.median())


def share_by_value(series: pd.Series, values: list[str], prefix: str) -> dict[str, float]:
    if series.empty:
        return {f"{prefix}_{value}_share": 0.0 for value in values}
    normalized = series.astype("string").fillna("missing")
    return {
        f"{prefix}_{value}_share": float(normalized.eq(value).mean())
        for value in values
    }


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "sequential_goal_forecasting_geometry"
DATA_INTAKE_PATH = PROJECT_ROOT / "src" / "python" / "data_intake.py"

spec = util.spec_from_file_location("project_data_intake", DATA_INTAKE_PATH)
data_intake = util.module_from_spec(spec)
assert spec is not None and spec.loader is not None
sys.modules[spec.name] = data_intake
spec.loader.exec_module(data_intake)

normalize_dataframe = data_intake.normalize_dataframe
NA_TOKENS = data_intake.NA_TOKENS


In [2]:
sequence_feature_path = ARTIFACT_DIR / "sequence_feature_table.csv"
if not sequence_feature_path.exists():
    raise FileNotFoundError(
        "Missing artifacts/sequential_goal_forecasting_geometry/sequence_feature_table.csv. "
        "Run the geometry notebook first or regenerate that artifact."
    )

knn_df = pd.read_csv(DATA_DIR / "knn_dataset.csv")
players_df = load_normalized_table("players_quarters_final.csv")
pressure_df = add_event_time(load_normalized_table("player_appearance_behaviour_under_pressure.csv"))
run_df = add_event_time(load_normalized_table("player_appearance_run.csv"))
shot_df = add_event_time(load_normalized_table("player_appearance_shot_limited.csv"))
geometry_df = pd.read_csv(sequence_feature_path)

row_link = players_df[
    ["player_appearance_id", "player_id", "fixture_id", "checkpoint"]
].copy()
row_link["checkpoint_cont_min"] = row_link["checkpoint"].map(CHECKPOINT_TO_CONT_MIN)
row_link["scored_after_eval_key"] = (
    row_link["fixture_id"].astype("string")
    + "__"
    + row_link["checkpoint"].astype("string")
    + "__"
    + row_link["player_id"].astype("string")
)

duplicate_eval_keys = int(row_link["scored_after_eval_key"].duplicated().sum())
if duplicate_eval_keys != 0:
    raise ValueError(f"Duplicate eval keys in players_quarters_final.csv: {duplicate_eval_keys}")

working_df = knn_df.merge(
    row_link[["scored_after_eval_key", "player_appearance_id", "fixture_id", "checkpoint_cont_min"]],
    on="scored_after_eval_key",
    how="left",
    validate="one_to_one",
)

missing_row_links = int(working_df["player_appearance_id"].isna().sum())
if missing_row_links != 0:
    raise ValueError(f"Missing row links for {missing_row_links} KNN rows")

geometry_feature_columns = [
    column
    for column in geometry_df.columns
    if column.startswith("seq15m_") or column.startswith("seqpos_")
]
geometry_export = geometry_df.merge(
    row_link[
        ["scored_after_eval_key", "player_appearance_id", "fixture_id", "checkpoint", "checkpoint_cont_min"]
    ],
    on=["player_appearance_id", "fixture_id", "checkpoint", "checkpoint_cont_min"],
    how="left",
    validate="one_to_one",
)
geometry_export = geometry_export[["scored_after_eval_key", *geometry_feature_columns]].copy()
working_df = working_df.merge(
    geometry_export,
    on="scored_after_eval_key",
    how="left",
    validate="one_to_one",
)

pressure_groups = {
    player_appearance_id: frame.sort_values(["event_cont_min", "id"]).reset_index(drop=True)
    for player_appearance_id, frame in pressure_df.groupby("player_appearance_id")
}
run_groups = {
    player_appearance_id: frame.sort_values(["event_cont_min", "id"]).reset_index(drop=True)
    for player_appearance_id, frame in run_df.groupby("player_appearance_id")
}
shot_groups = {
    player_appearance_id: frame.sort_values(["event_cont_min", "id"]).reset_index(drop=True)
    for player_appearance_id, frame in shot_df.groupby("player_appearance_id")
}


In [3]:
extra_feature_rows = []

for row in working_df[["scored_after_eval_key", "player_appearance_id", "checkpoint_cont_min"]].itertuples(index=False):
    checkpoint_time = int(row.checkpoint_cont_min)
    history_start = max(1, checkpoint_time - LOOKBACK_MINUTES + 1)
    player_appearance_id = int(row.player_appearance_id)

    player_pressure = pressure_groups.get(player_appearance_id)
    player_runs = run_groups.get(player_appearance_id)
    player_shots = shot_groups.get(player_appearance_id)

    pressure_last15 = (
        player_pressure[
            (player_pressure["event_cont_min"] >= history_start)
            & (player_pressure["event_cont_min"] <= checkpoint_time)
        ].copy()
        if player_pressure is not None
        else pd.DataFrame(columns=pressure_df.columns)
    )
    runs_last15 = (
        player_runs[
            (player_runs["event_cont_min"] >= history_start)
            & (player_runs["event_cont_min"] <= checkpoint_time)
        ].copy()
        if player_runs is not None
        else pd.DataFrame(columns=run_df.columns)
    )
    shots_last15 = (
        player_shots[
            (player_shots["event_cont_min"] >= history_start)
            & (player_shots["event_cont_min"] <= checkpoint_time)
        ].copy()
        if player_shots is not None
        else pd.DataFrame(columns=shot_df.columns)
    )

    angle_series = pressure_last15["pass_angle"].dropna().astype(float)
    run_possession_series = runs_last15["possession"].dropna()
    ordered_run_possessions = run_possession_series.astype(str).tolist()
    run_possession_transition_count = 0
    if ordered_run_possessions:
        run_possession_transition_count = sum(
            current != previous
            for previous, current in zip(ordered_run_possessions[:-1], ordered_run_possessions[1:])
        )

    shot_body_part_series = shots_last15["body_part"].dropna().astype(str)
    shot_technique_series = shots_last15["technique"].dropna().astype(str)
    shot_play_pattern_series = shots_last15["play_pattern"].dropna().astype(str)

    feature_row = {
        "scored_after_eval_key": row.scored_after_eval_key,
        "seqextra_pass_angle_observed_count": int(angle_series.shape[0]),
        "seqextra_pass_angle_mean_deg": safe_mean(angle_series),
        "seqextra_pass_angle_median_deg": safe_median(angle_series),
        "seqextra_pass_angle_std_deg": safe_std(angle_series),
        "seqextra_pass_angle_last_deg": float(angle_series.iloc[-1]) if not angle_series.empty else 0.0,
        "seqextra_run_distinct_possessions": int(run_possession_series.nunique()),
        "seqextra_run_possession_transition_count": int(run_possession_transition_count),
        "seqextra_run_last_stage": last_non_null(runs_last15["stage"]),
        "seqextra_shot_last_body_part": last_non_null(shots_last15["body_part"]),
        "seqextra_shot_last_technique": last_non_null(shots_last15["technique"]),
        "seqextra_shot_last_play_pattern": last_non_null(shots_last15["play_pattern"]),
    }
    feature_row.update(share_by_value(runs_last15["stage"], RUN_STAGE_VALUES, "seqextra_run_stage"))
    feature_row.update(share_by_value(shots_last15["body_part"], SHOT_BODY_PART_VALUES, "seqextra_shot_body_part"))
    feature_row.update(share_by_value(shots_last15["technique"], SHOT_TECHNIQUE_VALUES, "seqextra_shot_technique"))
    feature_row.update(share_by_value(shots_last15["play_pattern"], SHOT_PLAY_PATTERN_VALUES, "seqextra_shot_play_pattern"))
    extra_feature_rows.append(feature_row)

extra_features_df = pd.DataFrame(extra_feature_rows)
sequencing_df = working_df.merge(
    extra_features_df,
    on="scored_after_eval_key",
    how="left",
    validate="one_to_one",
)

sequencing_df = sequencing_df.drop(columns=["player_appearance_id", "fixture_id", "checkpoint_cont_min"])

missing_geometry_rows = int(sequencing_df[geometry_feature_columns].isna().any(axis=1).sum())
if missing_geometry_rows != 0:
    raise ValueError(f"Missing geometry features for {missing_geometry_rows} rows")


In [4]:
export_path = DATA_DIR / "sequencing_dataset.csv"
sequencing_df.to_csv(export_path, index=False)

print(f"Saved sequencing_dataset.csv to: {export_path}")
print(f"shape = {sequencing_df.shape}")
print("added geometry columns:", len(geometry_feature_columns))
print("added extra event columns:", extra_features_df.shape[1] - 1)
sequencing_df.head()


Saved sequencing_dataset.csv to: /Users/jan/Documents/competitions/hackatons/WEC_26/data/sequencing_dataset.csv
shape = (3486, 99)
added geometry columns: 38
added extra event columns: 30


,player_id,checkpoint,position,is_home,formation,subbed,jersey_number,last15_sprints,last15_hsr,last15_shots,...,seqextra_shot_technique_other_share,seqextra_shot_technique_overhead_kick_share,seqextra_shot_technique_lob_share,seqextra_shot_play_pattern_regular_play_share,seqextra_shot_play_pattern_corner_kick_share,seqextra_shot_play_pattern_counter_attack_share,seqextra_shot_play_pattern_indirect_free_kick_share,seqextra_shot_play_pattern_direct_free_kick_share,seqextra_shot_play_pattern_throw_in_share,seqextra_shot_play_pattern_penalty_share
0,2702,H1_15,G,True,4-1-4-1,False,1,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2732,H1_15,D,True,4-1-4-1,False,2,2,5,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3296,H1_15,D,True,4-1-4-1,False,4,2,4,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3297,H1_15,D,True,4-1-4-1,False,5,1,5,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,3298,H1_15,A,True,4-1-4-1,True,6,4,15,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
